# Exercise 2.6 — Pandas Joins and Merges — SOLUTIONS

This notebook uses the Zambia ECON 2025 establishment census (10% sample) and fake supplementary tables to practice:
- `pd.merge` (left, inner) with different key types
- `pd.concat` for stacking DataFrames
- Join QA: checking for key mismatches, NaN rates, and silent row multiplication

### Setup (run first)

In [ ]:
import os
import pandas as pd
import pyreadstat

DATA_RAW    = '../../data/0_raw/zambia'
DATA_OUT    = '../../data/02_processed'

sav_path = os.path.join(DATA_RAW, 'ECON2025_10percent_120326.sav')
print('Exists?', os.path.exists(sav_path))

df_raw, meta = pyreadstat.read_sav(sav_path, encoding='latin1')

# Keep only the columns we need for this exercise
COLS = ['interview__key', 'G1_prov', 'G2_dist', 'isic_2dgts', 'isic4']
df = df_raw[COLS].copy()

# G1_prov comes in as float — convert to int for cleaner keys
df['G1_prov'] = df['G1_prov'].astype('Int64')   # nullable integer
df['isic_2dgts'] = df['isic_2dgts'].astype('Int64')

print(f'Shape: {df.shape}')
df.head()

---

## Task 1 — Create and merge a province lookup table

In [ ]:
prov_lookup = pd.DataFrame({
    'prov_code': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'province_name': [
        'Central', 'Copperbelt', 'Eastern', 'Luapula', 'Lusaka',
        'Muchinga', 'Northern', 'North-Western', 'Southern', 'Western'
    ],
    'region': [
        'Central', 'Copperbelt', 'Eastern', 'Northern', 'Lusaka',
        'Northern', 'Northern', 'Western', 'Southern', 'Western'
    ]
})

# QA before merging
print('Left keys missing :', df['G1_prov'].isna().sum())
print('Left keys unique  :', df['G1_prov'].nunique())
print('Right keys unique :', prov_lookup['prov_code'].nunique())

In [ ]:
rows_before = len(df)

df_merged = pd.merge(df, prov_lookup, left_on='G1_prov', right_on='prov_code', how='left')

rows_after = len(df_merged)
print('Rows before:', rows_before, '| after:', rows_after)
if rows_after != rows_before:
    print('⚠️  Row count changed by', rows_after - rows_before)

df_merged[['G1_prov', 'province_name', 'region']].head(8)

In [ ]:
# How many rows have a NaN in province_name after the merge?
print('NaN in province_name:', df_merged['province_name'].isna().sum())

**Answers:**

- **Row count:** it should not change. A left join keeps every row from the left table exactly once — as long as the right-table key is unique (no duplicates in `prov_lookup`), there is a one-to-one or zero-to-one match per left row.
- **NaN in `province_name`:** any row whose `G1_prov` value is null (or has no matching code in the lookup) will get `NaN`. Those are establishments with missing province information in the raw data.

---

## Task 2 — Create and merge an ISIC description lookup

In [ ]:
isic_lookup = pd.DataFrame({
    'isic_2d': [1, 10, 25, 41, 45, 46, 47, 55, 56, 62],
    'isic_description': [
        'Crop production', 'Food products', 'Fabricated metals',
        'Building construction', 'Motor vehicle trade', 'Wholesale trade',
        'Retail trade', 'Accommodation', 'Food & beverage service',
        'Computer programming'
    ]
})

# QA
print('ISIC lookup unique codes:', isic_lookup['isic_2d'].nunique())
print('Main data unique codes  :', df_merged['isic_2dgts'].nunique())

In [ ]:
df_merged = pd.merge(df_merged, isic_lookup, left_on='isic_2dgts', right_on='isic_2d', how='left')

print(df_merged['isic_description'].value_counts().head(10))

In [ ]:
# Match rate
match_rate = df_merged['isic_description'].notna().mean()
print(f'Match rate: {match_rate:.1%}')
print(f'No match  : {1 - match_rate:.1%}')

**Answers:**

- The lookup covers only 10 out of many 2-digit ISIC codes in the data, so most rows will be `NaN`. That is expected — the lookup is partial by design.
- A `left` join is correct because we want to keep **all** establishments regardless of whether their ISIC code has a description. An `inner` join would silently drop every unmatched row, reducing the dataset.

---

## Task 3 — Split and concatenate

In [ ]:
df_lusaka = df[df['G1_prov'] == 5].copy()   # province 5 = Lusaka
df_other  = df[df['G1_prov'] != 5].copy()

print(f'Lusaka: {len(df_lusaka)} | Other: {len(df_other)} | Total: {len(df)}')

In [ ]:
df_combined = pd.concat([df_lusaka, df_other], ignore_index=True)

print(f'Combined: {len(df_combined)} | Original: {len(df)}')
if len(df_combined) != len(df):
    print('⚠️  Row counts do not match!', len(df_combined) - len(df))

**Answers:**

- **`ignore_index=True`:** both sub-DataFrames share the same integer index from the original `df`. Without `ignore_index=True`, the combined index would have duplicate values (0, 1, 2 … from both batches), which can cause silent bugs in downstream `.loc` calls. Resetting the index gives a clean 0-to-N sequence.
- **Different columns:** `pd.concat` aligns on column names. Missing columns in either piece are filled with `NaN`. Columns that exist in one batch but not the other will appear in the result with `NaN` for all rows from the batch that lacked them — a common source of confusion.

---

## Task 4 — Join QA: detect duplicates created by a lookup table

In [ ]:
prov_lookup_bad = pd.DataFrame({
    'prov_code': [1, 1, 2, 3],
    'province_name': ['Central', 'Central (duplicate)', 'Copperbelt', 'Eastern'],
    'region': ['Central', 'Central', 'Copperbelt', 'Eastern'],
})

print('Duplicate keys in bad lookup:', prov_lookup_bad['prov_code'].duplicated().sum())
print(prov_lookup_bad[prov_lookup_bad['prov_code'].duplicated(keep=False)])

In [ ]:
rows_before = len(df)
df_test = pd.merge(df, prov_lookup_bad, left_on='G1_prov', right_on='prov_code', how='left')
rows_after = len(df_test)

print('Rows before:', rows_before, '| after:', rows_after)
if rows_after != rows_before:
    print('⚠️  Row count changed by', rows_after - rows_before)

In [ ]:
prov_lookup_fixed = prov_lookup_bad.drop_duplicates(subset=['prov_code'], keep='first')

rows_before = len(df)
df_test_fixed = pd.merge(df, prov_lookup_fixed, left_on='G1_prov', right_on='prov_code', how='left')
rows_after = len(df_test_fixed)

print('Rows before:', rows_before, '| after:', rows_after)

**Answers:**

- **Why did rows multiply?** Pandas performs a cross-join for each matching pair. One left row with `G1_prov == 1` matched **two** rows in `prov_lookup_bad`, so it was duplicated in the output. The row count increases by exactly (number of left rows with `G1_prov == 1`), i.e. the count of Central establishments.
- **Fix at source vs `drop_duplicates`:** `drop_duplicates` is a defensive patch, not a fix. In production, the duplicate in the reference table is almost certainly a data-entry error. The right approach is to fix it upstream (in the source system or in the ETL that produces the lookup) and document why. Using `drop_duplicates` silently hides the issue from everyone downstream.

---

## Task 5 — Build fake trade data and practice concatenation + many-to-one join

In [ ]:
imports = pd.DataFrame({
    'year':        [2024, 2024, 2025, 2025, 2025],
    'partner':     ['ZAF', 'TZA', 'ZAF', 'MOZ', 'TZA'],
    'hs2':         ['10', '10', '10', '12', '12'],
    'value_local': [120000, 80000, 150000, 50000, 40000],
})
imports['flow'] = 'import'

exports = pd.DataFrame({
    'year':        [2024, 2024, 2025, 2025],
    'partner':     ['ZAF', 'COD', 'ZAF', 'COD'],
    'hs2':         ['10', '10', '12', '10'],
    'value_local': [60000, 30000, 70000, 45000],
})
exports['flow'] = 'export'

trade = pd.concat([imports, exports], ignore_index=True)
print(trade)

In [ ]:
hs_lookup = pd.DataFrame({
    'hs2':     ['10', '12'],
    'product': ['Cereals', 'Oil seeds'],
})

n_dups = hs_lookup['hs2'].duplicated().sum()
print('HS lookup duplicate keys:', n_dups)

trade = pd.merge(trade, hs_lookup, on='hs2', how='left')
print(trade)

**Answers:**

- **Many-to-one:** `trade` has multiple rows per `hs2` code (e.g. several imports with `hs2='10'`), while `hs_lookup` has exactly one row per code. Each left row finds at most one match — the product description is broadcast to all matching trade rows.
- **Detecting duplicates:** always run `hs_lookup['hs2'].duplicated().sum()` before merging. If the count is > 0, print the offending rows and investigate before proceeding. Alternatively, assert `assert hs_lookup['hs2'].is_unique` and let the test fail loudly.

---

## Task 6 — Multi-key join: currency conversion by year

In [ ]:
fx = pd.DataFrame({
    'year':          [2024, 2024, 2025, 2025],
    'currency':      ['MWK', 'ZMW', 'MWK', 'ZMW'],
    'usd_per_local': [0.00058, 0.041, 0.00055, 0.039],
})

trade['currency'] = 'ZMW'

print('FX duplicate (year, currency):', fx.duplicated(['year', 'currency']).sum())

In [ ]:
trade_fx = pd.merge(trade, fx, on=['year', 'currency'], how='left')

print('Rows missing FX rate:', trade_fx['usd_per_local'].isna().sum())

trade_fx['value_usd'] = trade_fx['value_local'] * trade_fx['usd_per_local']

print(f'trade: {len(trade)} | trade_fx: {len(trade_fx)}')

trade_fx

**Answers:**

- **Why multi-key?** The FX rate changes every year. If we join only on `currency`, every 2024 row would match both the 2024 and 2025 rate, doubling rows. Joining on `['year', 'currency']` identifies one unique rate per (year, currency) combination.
- **Duplicate (2025, ZMW) rows:** the merge would fan out — every 2025 ZMW trade row would be duplicated. Detect it with `fx.duplicated(['year', 'currency']).sum()` and assert it is zero before merging. In production, this check should be part of your data validation pipeline.

---

## Task 7 — Export the enriched dataset

In [ ]:
df_final = df_merged.reset_index(drop=True)

out_path = os.path.join(DATA_OUT, 'econ2025_analysis_ready.csv')
df_final.to_csv(out_path, index=False)
print(f'Saved: {out_path} ({df_final.shape[0]} rows, {df_final.shape[1]} cols)')

## Summary

This notebook enriched the ECON 2025 establishment dataset through a series of left joins:

- **Province lookup** (`prov_lookup`): added human-readable province names and broad regions to each establishment using the `G1_prov` code. A left join ensures no establishment is dropped even if its province code is missing.
- **ISIC lookup** (`isic_lookup`): added a plain-language activity description for a subset of 2-digit ISIC codes. Most rows remain `NaN` because the lookup is intentionally partial — a left join is the only correct choice here.

Key QA steps performed before every merge:
1. Check for null values in the join key on the left.
2. Assert that the right-table key is unique (to prevent silent row multiplication).
3. Compare row counts before and after — any change signals a problem.